In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy.stats import chi2_contingency

df = pd.read_csv("marketing_campaign.csv")

# Cleaning steps
# 1. Impute missing 'Income' with the median
median_income = df['Income'].median()
df['Income'].fillna(median_income, inplace=True)

# 2. Convert 'Dt_Customer' to datetime
df['Dt_Customer'] = pd.to_datetime(df['Dt_Customer'], format='%d-%m-%Y')

# Display the first few rows and column information of the cleaned DataFrame
print(df.head().to_markdown(index=False, numalign="left", stralign="left"))
print(df.info())

# 1. Create a new column: Total Spending (Amount Spent)
# Select all columns starting with 'Mnt' and sum them row-wise
mnt_cols = [col for col in df.columns if col.startswith('Mnt')]
df['AmountSpent'] = df[mnt_cols].sum(axis=1)

# Summary Statistics for the new feature
print("Summary Statistics for Total Amount Spent:")
print(df['AmountSpent'].describe().to_markdown(numalign="left", stralign="left"))

# 2. Visualize Distribution: Histogram of Total Amount Spent
plt.figure(figsize=(10, 6))
plt.hist(df['AmountSpent'], bins=50, edgecolor='black', color='skyblue')
plt.title('Distribution of Total Amount Spent by Customers')
plt.xlabel('Total Amount Spent (USD)')
plt.ylabel('Number of Customers')
plt.grid(axis='y', alpha=0.7)
plt.axvline(df['AmountSpent'].median(), color='red', linestyle='dashed', linewidth=1, label=f'Median: ${df["AmountSpent"].median():,.0f}')
plt.axvline(df['AmountSpent'].mean(), color='green', linestyle='dashed', linewidth=1, label=f'Mean: ${df["AmountSpent"].mean():,.0f}')
plt.legend()
plt.tight_layout()
plt.close()

# List of numerical columns to plot (excluding ID and constant/less informative Z columns)
numeric_cols = df.select_dtypes(include=np.number).columns.drop(['ID', 'Z_CostContact', 'Z_Revenue']).tolist()

# Define the figure size and subplot grid
n_cols = 5
n_rows = int(np.ceil(len(numeric_cols) / n_cols))
plt.figure(figsize=(20, 4 * n_rows))

# Plot histograms/KDE for each numerical column
for i, col in enumerate(numeric_cols):
    plt.subplot(n_rows, n_cols, i + 1)

    if len(df[col].unique()) > 15:
        sns.histplot(df[col], kde=True, bins=30, color='skyblue', edgecolor='black')
    else:
        sns.histplot(df[col], bins=len(df[col].unique()), discrete=True, color='lightcoral', edgecolor='black',
                     shrink=0.8)

    plt.title(col, fontsize=12)
    plt.xlabel("")
    plt.ylabel("")

plt.suptitle('Distribution of Numerical Variables (Histograms and KDEs)', fontsize=18, y=1.02)
plt.tight_layout()
plt.close()

# Select categorical columns
categorical_cols = ['Education', 'Marital_Status']

# Set up the figure and axes
plt.figure(figsize=(14, 6))

# Plot 1: Education Distribution
plt.subplot(1, 2, 1)
# Calculate value counts and sort them for plotting
education_counts = df['Education'].value_counts().reset_index()
education_counts.columns = ['Education', 'Count']
sns.barplot(
    data=education_counts,
    x='Education',
    y='Count',
    palette='viridis'
)
plt.title('Distribution of Customer Education Level', fontsize=14)
plt.xlabel('Education Level')
plt.ylabel('Number of Customers')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.5)

# Plot 2: Marital Status Distribution

plt.subplot(1, 2, 2)
# Calculate value counts and sort them for plotting
marital_counts = df['Marital_Status'].value_counts().reset_index()
marital_counts.columns = ['Marital_Status', 'Count']
# Sort by count (descending)
marital_counts = marital_counts.sort_values(by='Count', ascending=False)
sns.barplot(
    data=marital_counts,
    x='Marital_Status',
    y='Count',
    palette='magma'
)
plt.title('Distribution of Customer Marital Status', fontsize=14)
plt.xlabel('Marital Status')
plt.ylabel('Number of Customers')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.5)

plt.tight_layout()
plt.close()

# Columns suitable for Box Plots (excluding binary and low-cardinality count columns)
boxplot_cols = [
    'Year_Birth', 'Age', 'Income', 'Recency',
    'MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts',
    'MntSweetProducts', 'MntGoldProds', 'AmountSpent',
    'NumDealsPurchases', 'NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth'
]
# Define the figure size and subplot grid
n_cols = 4
n_rows = int(np.ceil(len(boxplot_cols) / n_cols))
plt.figure(figsize=(18, 4 * n_rows))
# Plot box plots for each selected column
for i, col in enumerate(boxplot_cols):
    plt.subplot(n_rows, n_cols, i + 1)

    # Use seaborn boxplot
    sns.boxplot(y=df[col], color='lightcoral')

    plt.title(f'Box Plot of {col}', fontsize=12)
    plt.ylabel(col)
    plt.xlabel("")

plt.suptitle('Outlier Identification using Box Plots for Numerical Variables', fontsize=16, y=1.02)
plt.tight_layout()
plt.close()

# Select all relevant numerical columns for correlation analysis
numeric_cols_for_corr = df.select_dtypes(include=np.number).columns.drop(['ID', 'Z_CostContact', 'Z_Revenue']).tolist()
corr_df = df[numeric_cols_for_corr]

# 1. Calculate the Correlation Matrix
correlation_matrix = corr_df.corr()

# 2. Extract and sort correlations with the target variable 'Response'
response_corr = correlation_matrix['Response'].sort_values(ascending=False).drop('Response')

# 3. Visualize the full correlation matrix using a Heatmap
plt.figure(figsize=(18, 16))
sns.heatmap(
    correlation_matrix,
    annot=False,
    cmap='coolwarm',
    fmt=".2f",
    linewidths=.5,
    linecolor='black'
)
plt.title('Correlation Matrix of All Numerical Variables', fontsize=18)
plt.tight_layout()
plt.close()

# Print the sorted correlation table for analysis
print("\n--- Correlation with Target Variable (Response) ---")
print(response_corr.to_markdown(numalign="left", stralign="left"))

# --- Feature Cleaning: Marital Status Grouping ---
# Group low-frequency categories into 'Single' for robust statistical testing/visualization
df['Marital_Status_Clean'] = df['Marital_Status'].replace({
    'Alone': 'Single',
    'Absurd': 'Single',
    'YOLO': 'Single'
})

# --- 1. Statistical Exploration: Chi-Square Test ---
print("--- Chi-Square Test Results (Relationship with Response) ---")

# Test 1: Education vs Response
contingency_table_edu = pd.crosstab(df['Education'], df['Response'])
chi2_edu, p_edu, dof_edu, expected_edu = chi2_contingency(contingency_table_edu)
print(f"Education vs Response: Chi2 = {chi2_edu:.2f}, p-value = {p_edu:.5f}")

# Test 2: Cleaned Marital Status vs Response
contingency_table_marital = pd.crosstab(df['Marital_Status_Clean'], df['Response'])
chi2_marital, p_marital, dof_marital, expected_marital = chi2_contingency(contingency_table_marital)
print(f"Marital_Status_Clean vs Response: Chi2 = {chi2_marital:.2f}, p-value = {p_marital:.5f}")

# --- 2. Visualization: Bar Plots of Response Rate ---

# Calculate response rate (mean of Response) for each category
edu_response_rate = df.groupby('Education')['Response'].mean().sort_values(ascending=False).reset_index()
marital_response_rate = df.groupby('Marital_Status_Clean')['Response'].mean().sort_values(ascending=False).reset_index()

# Set up the figure and axes
plt.figure(figsize=(14, 6))

#Plot 1: Education Response Rate
plt.subplot(1, 2, 1)
sns.barplot(
    data=edu_response_rate,
    x='Education',
    y='Response',
    palette='Pastel1'
)
plt.title('Campaign Response Rate by Education Level', fontsize=14)
plt.xlabel('Education Level')
plt.ylabel('Response Rate (Mean)')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.5)

# Plot 2: Marital Status Response Rate
plt.subplot(1, 2, 2)
sns.barplot(
    data=marital_response_rate,
    x='Marital_Status_Clean',
    y='Response',
    palette='Pastel2'
)
plt.title('Campaign Response Rate by Marital Status (Cleaned)', fontsize=14)
plt.xlabel('Marital Status')
plt.ylabel('Response Rate (Mean)')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.5)

plt.tight_layout()
plt.show()
plt.close()

# Clean Marital Status (Group low-frequency categories into 'Single')
df['Marital_Status_Clean'] = df['Marital_Status'].replace({
    'Alone': 'Single',
    'Absurd': 'Single',
    'YOLO': 'Single'
})

# Create 'AmountSpent' feature
mnt_cols = [col for col in df.columns if col.startswith('Mnt')]
df['AmountSpent'] = df[mnt_cols].sum(axis=1)

# --- Visualization: Box Plots of Numerical vs Categorical ---

# Set up the figure and axes for 4 plots
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
sns.set_palette("viridis")

# Plot 1: Income vs Education
sns.boxplot(ax=axes[0, 0], x='Education', y='Income', data=df)
axes[0, 0].set_title('Income Distribution by Education Level', fontsize=14)
axes[0, 0].tick_params(axis='x', rotation=45)
axes[0, 0].set_xlabel('Education Level')
axes[0, 0].set_ylabel('Income (USD)')

# Plot 2: AmountSpent vs Education
sns.boxplot(ax=axes[0, 1], x='Education', y='AmountSpent', data=df)
axes[0, 1].set_title('Total Spending Distribution by Education Level', fontsize=14)
axes[0, 1].tick_params(axis='x', rotation=45)
axes[0, 1].set_xlabel('Education Level')
axes[0, 1].set_ylabel('Total Amount Spent (USD)')

# Plot 3: Income vs Marital Status
sns.boxplot(ax=axes[1, 0], x='Marital_Status_Clean', y='Income', data=df)
axes[1, 0].set_title('Income Distribution by Marital Status', fontsize=14)
axes[1, 0].tick_params(axis='x', rotation=45)
axes[1, 0].set_xlabel('Marital Status')
axes[1, 0].set_ylabel('Income (USD)')

# Plot 4: AmountSpent vs Marital Status
sns.boxplot(ax=axes[1, 1], x='Marital_Status_Clean', y='AmountSpent', data=df)
axes[1, 1].set_title('Total Spending Distribution by Marital Status', fontsize=14)
axes[1, 1].tick_params(axis='x', rotation=45)
axes[1, 1].set_xlabel('Marital Status')
axes[1, 1].set_ylabel('Total Amount Spent (USD)')

plt.suptitle('Relationship Between Key Numerical and Categorical Variables', fontsize=16, y=1.02)
plt.tight_layout()
plt.close()






















































































